# 05 – SQL analysis
Business questions answered with SQL on the `complaints` table (SQLite).

In [5]:
import sqlite3
import pandas as pd

pd.set_option("display.max_colwidth", None)

conn = sqlite3.connect("../data/processed/complaints.db")

def q(sql):
    return pd.read_sql(sql, conn)
    

## Q1 – How has complaint volume changed over time?

In [2]:
q("""
SELECT
    year,
    COUNT(*) AS complaints
FROM complaints
GROUP BY year
ORDER BY year;
""")

,year,complaints
0,2023,93778
1,2024,106630
2,2025,152520
3,2026,158467


**Insight:** Complaints rose by about 63% from 2023 to 2025. Based on the pace up to 14 November 2026, complaints are projected to reach roughly 182,000 by year-end, approximately 19% higher than in 2025. However, rising complaint volumes do not necessarily indicate poorer service; they may also reflect a growing customer base or increased awareness of the CFPB. This makes it important to examine the types of complaints customers raise and how banks respond. The product-level analysis also reveals that checking and savings account complaints increased by 59% from 2024 to 2025, contributing significantly to the overall rise, while credit card complaints in 2026 have already exceeded the full-year 2025 total, indicating that credit cards are a growing area of concern.


In [3]:
q("""
SELECT
    year,
    product,
    COUNT(*) AS complaints
FROM complaints
GROUP BY year, product
ORDER BY year, product;
""")

,year,product,complaints
0,2023,Checking or savings account,49794
1,2023,Credit card,43984
2,2024,Checking or savings account,52652
3,2024,Credit card,53978
4,2025,Checking or savings account,83747
5,2025,Credit card,68773
6,2026,Checking or savings account,78460
7,2026,Credit card,80007


## Q2 – What are customers complaining about most?

In [6]:
q("""
SELECT
    issue,
    COUNT(*) AS complaints
FROM complaints
WHERE product = 'Checking or savings account'
GROUP BY issue
ORDER BY complaints DESC
LIMIT 5;
""")

,issue,complaints
0,Managing an account,149535
1,Closing an account,34623
2,Problem with a lender or other company charging your account,33239
3,Problem caused by your funds being low,23072
4,Opening an account,22430


In [7]:
q("""
SELECT
    issue,
    COUNT(*) AS complaints
FROM complaints
WHERE product = 'Credit card'
GROUP BY issue
ORDER BY complaints DESC
LIMIT 5;
""")

,issue,complaints
0,Problem with a purchase shown on your statement,75087
1,"Other features, terms, or problems",30704
2,Getting a credit card,30312
3,Fees or interest,27487
4,Closing your account,19024


In [8]:
q("""
SELECT
    issue,
    COUNT(*) AS complaints,
    ROUND(
        COUNT(*) * 100.0 /
        (SELECT COUNT(*)
         FROM complaints
         WHERE product = 'Checking or savings account'),
        2
    ) AS percentage
FROM complaints
WHERE product = 'Checking or savings account'
GROUP BY issue
ORDER BY complaints DESC
LIMIT 5;
""")

,issue,complaints,percentage
0,Managing an account,149535,56.50
1,Closing an account,34623,13.08
2,Problem with a lender or other company charging your account,33239,12.56
3,Problem caused by your funds being low,23072,8.72
4,Opening an account,22430,8.48


In [10]:
q("""
SELECT
    issue,
    COUNT(*) AS complaints,
    ROUND(
        COUNT(*) * 100.0 /
        (SELECT COUNT(*)
         FROM complaints
         WHERE product = 'Credit card'),
        2
    ) AS percentage
FROM complaints
WHERE product = 'Credit card'
GROUP BY issue
ORDER BY complaints DESC
LIMIT 5;
""")

,issue,complaints,percentage
0,Problem with a purchase shown on your statement,75087,30.43
1,"Other features, terms, or problems",30704,12.44
2,Getting a credit card,30312,12.28
3,Fees or interest,27487,11.14
4,Closing your account,19024,7.71


### Q2 – Drill-down: Managing an account

In [11]:
q("""
SELECT
    sub_issue,
    COUNT(*) AS complaints
FROM complaints
WHERE product = 'Checking or savings account'
AND issue = 'Managing an account'
GROUP BY sub_issue
ORDER BY complaints DESC
LIMIT 5;
""")

,sub_issue,complaints
0,Deposits and withdrawals,53470
1,Problem using a debit or ATM card,34728
2,Funds not handled or disbursed as instructed,17333
3,Banking errors,13856
4,Problem accessing account,11591


**Insight:** Checking and savings account complaints are mainly related to managing accounts, representing 56.5% of complaints for this product. A closer look shows that deposits and withdrawals are the biggest problem, accounting for approximately 36% of account-management complaints (53,470 complaints), followed by debit or ATM card issues with 34,728 complaints. In comparison, credit card complaints are more spread across different categories, with problems related to purchases shown on statements accounting for around 30% of complaints. Banks should prioritise improving deposit and withdrawal processes, resolving debit card issues, and making credit card transactions and billing more transparent to reduce customer complaints.